# model_stat_prototype

Statistiques du prototype. Trois bras comparés **sur la même population de triggers** :

| bras | origine |
|---|---|
| `model` | top-12 du modèle, classé sur tout le pool de la catégorie du trigger |
| `display` | 12 premiers de `relevantProducts` dans l'ordre d'affichage |
| `relevance` | 12 premiers de `relevantProducts` par `relevanceScore` décroissant |

Deux différences majeures avec `model_stat` :

- **Plus de bloc « full negatives » séparé.** L'ancien second bloc forçait `positives_prod` à un tableau vide, donc `prod_found` était toujours faux et `avg_recall_prod` toujours nul : la matrice 2×2 y était une tautologie. Ici `prod_found` est réellement calculé pour chaque trigger.
- **Le MRR de référence devient le coverage-adjusted.** L'ancien bloc principal ne contenait que des exécutions ayant au moins un positif dans le top-12 prod, ce qui rendait le matched-only comparable à `MRR_trigger`. La population n'est plus conditionnée à un succès de la prod, donc les triggers où personne ne trouve rien doivent compter zéro. Le matched-only reste affiché, mais comme métrique conditionnelle « quand ça marche, à quel rang ».

In [ ]:
%pip install /Workspace/Users/neil.braun@mirakl.com/.bundle/fast-gnn-benchmark/dev/files
dbutils.library.restartPython()

In [ ]:
import os
import boto3, json, io
import pyspark.sql.functions as F
from pyspark.sql import DataFrame

In [ ]:
BUCKET = "mirakl-data-science-tmp2"
PREFIX = "nbraun/datasets/coview-mdm"

sessions_raw_val_prototype = spark.read.parquet(f"s3://{BUCKET}/{PREFIX}/sessions_raw_val_prototype.parquet")
prod_results_val_prototype = spark.read.parquet(f"s3://{BUCKET}/{PREFIX}/prod_results_val_prototype.parquet")
model_results_val_prototype = spark.read.parquet(f"s3://{BUCKET}/{PREFIX}/model_results_val_prototype.parquet")

print(f"sessions_raw_val_prototype: {sessions_raw_val_prototype.count()} triggers")
display(sessions_raw_val_prototype.limit(1))

print(f"prod_results_val_prototype: {prod_results_val_prototype.count()} triggers")
display(prod_results_val_prototype.limit(1))

print(f"model_results_val_prototype: {model_results_val_prototype.count()} triggers")
display(model_results_val_prototype.limit(1))

## Métadonnées produits — union des produits de session, des deux variantes prod, du modèle, et des triggers

In [ ]:
product_ids = lambda df, col: (
    df.select(F.explode(col).alias("product")).select(F.col("product.internal_id").alias("internal_id"))
)

product_metadata_val_prototype = (
    sessions_raw_val_prototype
    .select(F.explode("session_products").alias("product"))
    .select(F.col("product.internal_id").alias("internal_id"))
    .union(product_ids(prod_results_val_prototype, "products_returned_display"))
    .union(product_ids(prod_results_val_prototype, "products_returned_relevance"))
    .union(product_ids(model_results_val_prototype, "products_returned"))
    .union(sessions_raw_val_prototype.select(F.col("trigger_internal_id").alias("internal_id")))
    .distinct()
)


def get_customer_db_name(customer_shortname: str) -> str:
    df_customer = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_gold_customer")
        .where(F.col("shortName") == customer_shortname)
        .select(F.col("databaseName").alias("db_name"))
    )
    return df_customer.collect()[0]["db_name"]


db_name = get_customer_db_name("maisons-du-monde")

df_products = (
    spark.table("mirakl_ai.ds_etl_prod.t2s_mongo_product_0_current")
    .filter(F.col("db_name") == db_name)
    .select(F.col("internalId").cast("bigint").alias("internal_id"), "name", "imageUrl")
    .dropDuplicates(["internal_id"])
)

product_metadata_val_prototype = (
    product_metadata_val_prototype
    .join(df_products, on="internal_id", how="left")
).cache()

print(f"produits distincts: {product_metadata_val_prototype.count()}")
print(f"produits non resolus dans le catalogue: {product_metadata_val_prototype.filter(F.col('name').isNull()).count()}")

display(product_metadata_val_prototype.limit(10))

In [ ]:
product_metadata_val_prototype.write.mode("overwrite").parquet(
    f"s3://{BUCKET}/{PREFIX}/product_metadata_val_prototype.parquet"
)
print("product_metadata_val_prototype.parquet uploaded")

## Assemblage des trois bras

`session_size` exclut le produit déclencheur lui-même, et sert de dénominateur identique aux trois bras.

In [ ]:
sessions_size_prototype = (
    sessions_raw_val_prototype
    .select(
        F.col("exec_code"),
        F.col("trigger_internal_id"),
        F.size(
            F.filter(
                F.col("session_products"),
                lambda p: p["internal_id"] != F.col("trigger_internal_id"),
            )
        ).alias("session_size"),
    )
)

display(sessions_size_prototype.limit(5))

In [ ]:
# Un exec_code absent de model_results (trigger hors graphe) donnerait des colonnes null, et
# F.size(null) vaut -1 en Spark -- ce qui polluerait silencieusement les moyennes. On marque donc
# le cas puis on remplace par des tableaux vides.
EMPTY_MODEL = F.array().cast("array<struct<internal_id:bigint,score:double,rank:int>>")

full_results = (
    prod_results_val_prototype
    .select(
        "exec_code",
        "trigger_internal_id",
        "products_returned_display", "positives_display", "negatives_display",
        "products_returned_relevance", "positives_relevance", "negatives_relevance",
    )
    .join(
        model_results_val_prototype.select(
            "exec_code",
            F.col("products_returned").alias("products_returned_model"),
            F.col("positives").alias("positives_model"),
            F.col("negatives").alias("negatives_model"),
        ),
        on="exec_code",
        how="left",
    )
    .join(sessions_size_prototype.select("exec_code", "session_size"), on="exec_code", how="left")
    .withColumn("missing_model", F.col("products_returned_model").isNull())
    .withColumn("products_returned_model", F.coalesce(F.col("products_returned_model"), EMPTY_MODEL))
    .withColumn("positives_model", F.coalesce(F.col("positives_model"), EMPTY_MODEL))
    .withColumn("negatives_model", F.coalesce(F.col("negatives_model"), EMPTY_MODEL))
    # un trigger sans categorie ou a pool vide a bien une ligne, mais un top-12 vide: il n'a pas
    # repondu, ce qui n'est pas la meme chose que repondre sans rien trouver.
    .withColumn("model_answered", F.size("products_returned_model") > 0)
).cache()

print(f"full_results: {full_results.count()} triggers")
print(f"  sans ligne modele        : {full_results.filter(F.col('missing_model')).count()}")
print(f"  modele n'a pas repondu   : {full_results.filter(~F.col('model_answered')).count()}")
print(f"  session vide (size <= 0) : {full_results.filter(F.col('session_size').isNull() | (F.col('session_size') <= 0)).count()}")

display(full_results.limit(1))

In [ ]:
ARMS = ["model", "display", "relevance"]

def add_metrics(df: DataFrame) -> DataFrame:
    for arm in ARMS:
        positives = F.col(f"positives_{arm}")
        df = df.withColumns({
            f"n_positives_{arm}": F.size(positives),
            f"found_{arm}": F.size(positives) > 0,
            f"recall_{arm}": F.when(
                F.col("session_size") > 0,
                F.size(positives) / F.col("session_size"),
            ),
            # rang du premier produit vu dans le top-12 du bras; null si aucun
            f"best_rank_{arm}": F.array_min(F.transform(positives, lambda p: p["rank"])),
        })
        df = df.withColumn(f"rr_{arm}", F.lit(1.0) / F.col(f"best_rank_{arm}"))
    return df


full_results_with_metrics = add_metrics(full_results).cache()

display(
    full_results_with_metrics.select(
        "exec_code", "session_size",
        *[c for arm in ARMS for c in (f"n_positives_{arm}", f"found_{arm}", f"best_rank_{arm}")],
    ).limit(5)
)

## Métriques agrégées

`MRR cov` (coverage-adjusted) est le chiffre de référence : les triggers sans aucun positif comptent zéro, dénominateur = tous les triggers. `MRR match` ne moyenne que sur les triggers ayant trouvé au moins un produit vu, et se lit « quand ce bras trouve, à quel rang ».

In [ ]:
agg_exprs = [
    F.count("*").alias("total_triggers"),
    F.count(F.when(F.col("recall_model").isNull(), 1)).alias("nb_recall_undefined"),
]
for arm in ARMS:
    agg_exprs += [
        F.avg(f"recall_{arm}").alias(f"avg_recall_{arm}"),
        F.avg(F.col(f"found_{arm}").cast("int")).alias(f"pct_found_{arm}"),
        F.avg(f"n_positives_{arm}").alias(f"avg_n_positives_{arm}"),
        F.sum(f"rr_{arm}").alias(f"sum_rr_{arm}"),
        F.count(f"best_rank_{arm}").alias(f"matched_{arm}"),
    ]

row = full_results_with_metrics.agg(*agg_exprs).collect()[0]
total = row["total_triggers"]

print(f"triggers: {total}")
print(f"recall non defini (session vide): {row['nb_recall_undefined']}")
print()
header = f"{'bras':12s} {'recall':>9s} {'found':>8s} {'n_pos':>7s} {'MRR cov':>9s} {'MRR match':>10s} {'matched':>9s}"
print(header)
print("-" * len(header))
for arm in ARMS:
    matched = row[f"matched_{arm}"]
    sum_rr = row[f"sum_rr_{arm}"] or 0.0
    print(
        f"{arm:12s} "
        f"{row[f'avg_recall_{arm}'] or 0:9.4f} "
        f"{row[f'pct_found_{arm}'] or 0:8.2%} "
        f"{row[f'avg_n_positives_{arm}'] or 0:7.3f} "
        f"{sum_rr / total if total else 0:9.4f} "
        f"{sum_rr / matched if matched else 0:10.4f} "
        f"{matched:9d}"
    )

## Deux matrices 2×2

`modele vs prod_display` et `modele vs prod_relevance`. Les triggers auxquels le modèle n'a pas pu répondre (pas de catégorie ou pool vide) sont exclus et comptés à part : les confondre avec « a répondu sans rien trouver » gonflerait artificiellement la case `prod_only`.

In [ ]:
all_cases = spark.createDataFrame(
    [("both_find",), ("model_only",), ("prod_only",), ("neither",)],
    ["case"],
)


def matrix_2x2(df: DataFrame, prod_arm: str) -> DataFrame:
    excluded = df.filter(~F.col("model_answered")).count()
    valid = df.filter(F.col("model_answered"))
    total_valid = valid.count()

    labeled = (
        all_cases
        .join(
            valid
            .withColumn(
                "case",
                F.when(F.col("found_model") & F.col(f"found_{prod_arm}"), "both_find")
                 .when(F.col("found_model") & ~F.col(f"found_{prod_arm}"), "model_only")
                 .when(~F.col("found_model") & F.col(f"found_{prod_arm}"), "prod_only")
                 .otherwise("neither"),
            )
            .groupBy("case")
            .count(),
            on="case",
            how="left",
        )
        .withColumn("count", F.coalesce(F.col("count"), F.lit(0)))
        .withColumn("pct", F.round(F.col("count") / total_valid * 100, 2))
        .orderBy("case")
    )

    print(f"=== modele vs prod_{prod_arm} — {total_valid} triggers valides "
          f"({excluded} exclus: modele n'a pas repondu) ===")
    display(labeled)
    return labeled


matrix_model_display = matrix_2x2(full_results_with_metrics, "display")
matrix_model_relevance = matrix_2x2(full_results_with_metrics, "relevance")

## Quels produits sont renvoyés

C'est la question de départ : les deux variantes prod renvoient-elles réellement des ensembles différents ? Si `pct_display_eq_relevance` est proche de 1, les deux bras prod sont interchangeables et les deux matrices ci-dessus doivent être quasi identiques.

In [ ]:
ids = lambda col: F.transform(col, lambda p: p["internal_id"])

overlap = full_results_with_metrics.select(
    F.size("products_returned_display").alias("n_display"),
    F.size("products_returned_relevance").alias("n_relevance"),
    F.size("products_returned_model").alias("n_model"),
    F.size(F.array_intersect(ids("products_returned_display"), ids("products_returned_relevance"))).alias("ov_display_relevance"),
    F.size(F.array_intersect(ids("products_returned_model"), ids("products_returned_display"))).alias("ov_model_display"),
    F.size(F.array_intersect(ids("products_returned_model"), ids("products_returned_relevance"))).alias("ov_model_relevance"),
)

overlap.agg(
    F.count("*").alias("total_triggers"),
    F.avg("n_display").alias("avg_n_display"),
    F.avg("n_relevance").alias("avg_n_relevance"),
    F.avg("n_model").alias("avg_n_model"),
    F.avg("ov_display_relevance").alias("avg_ov_display_relevance"),
    F.avg("ov_model_display").alias("avg_ov_model_display"),
    F.avg("ov_model_relevance").alias("avg_ov_model_relevance"),
    F.avg(
        ((F.col("ov_display_relevance") == F.col("n_display")) & (F.col("n_display") == F.col("n_relevance"))).cast("int")
    ).alias("pct_display_eq_relevance"),
).show(truncate=False)

## Contrôle d'alignement

Dans le prototype les triggers du modèle sont extraits de la table prod, donc l'alignement est garanti par construction. La vérification reste peu coûteuse et détecterait un croisement accidentel de parquets issus de deux runs différents.

In [ ]:
alignment_check = (
    prod_results_val_prototype.select("exec_code", F.col("trigger_internal_id").alias("tid_prod"))
    .join(
        model_results_val_prototype.select("exec_code", F.col("trigger_internal_id").alias("tid_model")),
        on="exec_code",
        how="inner",
    )
    .withColumn("match", F.col("tid_prod") == F.col("tid_model"))
    .groupBy("match")
    .count()
)

print(f"lignes comparees: {alignment_check.agg(F.sum('count')).first()[0]}")
display(alignment_check)

## Scores par type de session selon la prod

Pour chaque variante prod, la population est coupée en deux : les triggers où la prod a trouvé au moins un produit vu en session (`positive`), et ceux où elle n'a rien trouvé (`full_negative`). C'est le découpage que l'ancien pipeline matérialisait en deux tables disjointes, `prod_results_val` et `prod_negative_only_results_val` — il est ici une simple colonne calculée, donc disponible **par variante**.

**Une réserve de lecture.** Dans la partition `full_negative` d'une variante, les métriques *de cette variante* sont nulles par construction : `recall` à zéro et `best_rank` null, donc MRR nul. C'est la même tautologie que l'ancien bloc négatifs-seuls, sauf qu'ici elle est assumée. Le chiffre porteur d'information dans cette partition est celui du **modèle** — son taux de rattrapage là où la prod est passée à côté.

Deux filtres s'appliquent : `session_size > 0` écarte les sessions vides, qui tombent mécaniquement en `full_negative` pour les trois bras sans rien dire de personne, et `model_answered` écarte les triggers auxquels le modèle n'a pas répondu, qui compteraient sinon comme des échecs de prédiction.

In [ ]:
for arm in ["display", "relevance"]:
    print(f"===== type de session selon prod_{arm} =====")
    (
        full_results_with_metrics
        .filter((F.col("session_size") > 0) & F.col("model_answered"))
        .withColumn(
            "prod_session",
            F.when(F.col(f"found_{arm}"), "positive").otherwise("full_negative"),
        )
        .groupBy("prod_session")
        .agg(
            F.count("*").alias("triggers"),
            F.avg(F.col("found_model").cast("int")).alias("pct_model_found"),
            F.avg("recall_model").alias("avg_recall_model"),
            F.avg(f"recall_{arm}").alias(f"avg_recall_{arm}"),
            (F.sum("rr_model") / F.count("*")).alias("mrr_model_cov"),
            (F.sum(f"rr_{arm}") / F.count("*")).alias(f"mrr_{arm}_cov"),
        )
        .orderBy("prod_session")
        .show(truncate=False)
    )

## Croisement des deux variantes

La mesure la plus directe de la question de départ : est-ce que le choix des douze produits **change le verdict** ?

Les deux cases hors diagonale sont le cœur du sujet. `display_seul` compte les triggers où l'ordre d'affichage fait remonter un produit vu que le tri par `relevanceScore` aurait manqué ; `relevance_seul` compte l'inverse.

C'est plus fin que le `pct_display_eq_relevance` de la section précédente : deux ensembles de douze produits peuvent différer tout en aboutissant au même succès ou au même échec. Si les cases hors diagonale sont vides ou négligeables, les deux variantes sont interchangeables du point de vue du résultat, quelle que soit la différence entre les ensembles eux-mêmes. Si elles sont déséquilibrées, l'écart te dit quelle sélection est la meilleure et de combien.

In [ ]:
sess = (
    full_results_with_metrics
    .filter(F.col("session_size") > 0)
    .withColumn("sess_display", F.when(F.col("found_display"), "positive").otherwise("full_negative"))
    .withColumn("sess_relevance", F.when(F.col("found_relevance"), "positive").otherwise("full_negative"))
).cache()

total_sess = sess.count()
print(f"triggers avec session non vide: {total_sess}")

print("\ncomptages croises (lignes = display, colonnes = relevance):")
sess.groupBy("sess_display").pivot("sess_relevance", ["positive", "full_negative"]).count().show()

print("verdict par trigger:")
(
    sess
    .withColumn(
        "verdict",
        F.when(F.col("found_display") & F.col("found_relevance"), "les_deux_trouvent")
         .when(F.col("found_display") & ~F.col("found_relevance"), "display_seul")
         .when(~F.col("found_display") & F.col("found_relevance"), "relevance_seul")
         .otherwise("aucun_ne_trouve"),
    )
    .groupBy("verdict")
    .agg(
        F.count("*").alias("triggers"),
        F.round(F.count("*") / total_sess * 100, 3).alias("pct"),
    )
    .orderBy(F.col("triggers").desc())
    .show(truncate=False)
)

## Quatre matrices 2×2 par type de session

Deux variantes × deux types de session, le type étant défini par la variante elle-même : `positive` quand la prod trouve au moins un produit vu, `full_negative` sinon.

**Comment lire ces matrices.** `found_{variante}` est constant à l'intérieur de chaque partition, donc deux des quatre cases sont **structurellement nulles** : `model_only` et `neither` sur les sessions positives, `both_find` et `prod_only` sur les full negatives. Ces zéros sont attendus et servent de contrôle — la cellule imprime un avertissement s'ils ne le sont pas, ce qui signalerait un partitionnement cassé.

L'information qui varie est la répartition entre les deux cases restantes, c'est-à-dire le taux de réussite du modèle dans ce régime. C'est le même chiffre que `pct_model_found` de la section `by-session`, remis en forme de matrice pour retrouver la présentation de l'ancien notebook — où les deux blocs de stats donnaient exactement ces deux matrices, mais sur une seule définition des douze produits et avec un `prod_found` forcé à faux dans le second.

Les triggers auxquels le modèle n'a pas répondu et les sessions vides sont écartés, comme dans les sections précédentes.

In [ ]:
CASE_ORDER = ["both_find", "prod_only", "model_only", "neither"]

# Cases impossibles selon la partition: found_{prod_arm} y est constant.
FORCED_ZERO = {
    "positive": ["model_only", "neither"],
    "full_negative": ["both_find", "prod_only"],
}


def matrix_2x2_by_session(df: DataFrame, prod_arm: str, session_type: str) -> DataFrame:
    """Matrice modele x prod_{prod_arm}, restreinte au type de session defini par CETTE variante."""
    cond = F.col(f"found_{prod_arm}") if session_type == "positive" else ~F.col(f"found_{prod_arm}")

    counts = {
        r["case"]: r["count"]
        for r in (
            df
            .filter(F.col("model_answered") & (F.col("session_size") > 0) & cond)
            .withColumn(
                "case",
                F.when(F.col("found_model") & F.col(f"found_{prod_arm}"), "both_find")
                 .when(F.col("found_model") & ~F.col(f"found_{prod_arm}"), "model_only")
                 .when(~F.col("found_model") & F.col(f"found_{prod_arm}"), "prod_only")
                 .otherwise("neither"),
            )
            .groupBy("case")
            .count()
            .collect()          # 4 lignes au plus: une seule action par matrice
        )
    }
    total = sum(counts.values())

    print(f"=== modele vs prod_{prod_arm}  |  sessions {session_type}  ---  {total} triggers ===")
    if total == 0:
        print("   (partition vide)")
        return None

    unexpected = {c: counts[c] for c in FORCED_ZERO[session_type] if counts.get(c, 0) != 0}
    if unexpected:
        print(f"   ATTENTION: cases censees etre nulles dans cette partition -> {unexpected}")

    labeled = spark.createDataFrame(
        [
            (case, counts.get(case, 0), round(counts.get(case, 0) / total * 100, 2))
            for case in CASE_ORDER
        ],
        schema="case string, count long, pct double",
    )

    display(labeled)
    return labeled


session_matrices = {}
for prod_arm in ["display", "relevance"]:
    for session_type in ["positive", "full_negative"]:
        session_matrices[(prod_arm, session_type)] = matrix_2x2_by_session(
            full_results_with_metrics, prod_arm, session_type
        )

## Niveau session

`sessions_raw_val_prototype` ne porte pas `userId`, et `session_id` seul est un compteur **par utilisateur** — il ne peut donc pas servir de clé. On reconstruit une clé à partir du contenu de la session, identique pour tous les triggers qui la composent.

Deux précautions dans cette clé. `collect_list` ne garantit pas l'ordre entre groupes, et les triggers d'une même session sont des groupes distincts dans le `groupBy` qui a construit `session_products` : on **trie donc les identifiants avant de hacher**, sinon deux triggers de la même session produiraient deux clés différentes. Et on ajoute `session_id` ainsi que le premier `emitted` pour rendre les collisions entre sessions négligeables.

Ça reste un **proxy** : la vraie clé est `(userId, session_id)`, et `userId` a été perdu dans le `select` final de `graph_pipeline_prototype`. Le correctif propre est une ligne là-bas, au prix d'un nouveau run complet.

**Définition du type de session.** Une session est `positive` dès qu'**au moins un** de ses triggers a trouvé un produit vu — l'analogue naturel du niveau trigger. Remplacer `F.max` par `F.min` dans les agrégats donnerait la définition « tous ses triggers trouvent ».

In [ ]:
ids_of = lambda col: F.transform(col, lambda p: p["internal_id"])

session_key = F.xxhash64(
    F.col("session_id"),
    F.sort_array(ids_of(F.col("session_products"))),          # tri: collect_list n'ordonne pas
    F.array_min(F.transform("session_products", lambda p: p["emitted"])),
)

trigger_to_session = sessions_raw_val_prototype.select(
    "exec_code",
    session_key.alias("session_key"),
    F.size("session_products").alias("n_session_products"),   # identique pour tous les triggers
)

sessions_level = (
    trigger_to_session
    .join(
        full_results_with_metrics.select(
            "exec_code", "found_display", "found_relevance", "found_model", "model_answered",
        ),
        on="exec_code",
        how="inner",
    )
    .groupBy("session_key")
    .agg(
        F.count("*").alias("n_triggers"),
        F.max("n_session_products").alias("n_session_products"),
        # F.max -> "au moins un trigger trouve". F.min -> "tous les triggers trouvent".
        F.max(F.col("found_display").cast("int")).alias("any_display"),
        F.max(F.col("found_relevance").cast("int")).alias("any_relevance"),
        F.max(F.col("found_model").cast("int")).alias("any_model"),
        F.sum(F.col("found_display").cast("int")).alias("n_found_display"),
        F.sum(F.col("found_relevance").cast("int")).alias("n_found_relevance"),
    )
    .withColumn("sess_display", F.when(F.col("any_display") == 1, "positive").otherwise("full_negative"))
    .withColumn("sess_relevance", F.when(F.col("any_relevance") == 1, "positive").otherwise("full_negative"))
).cache()

n_sessions = sessions_level.count()
n_trig = trigger_to_session.count()

print(f"triggers (= executions prod) : {n_trig}")
print(f"sessions (proxy de cle)      : {n_sessions}")
print(f"triggers par session         : {n_trig / n_sessions:.2f}")
print()

for arm in ["display", "relevance"]:
    print(f"===== repartition des SESSIONS selon prod_{arm} =====")
    (
        sessions_level
        .groupBy(f"sess_{arm}")
        .agg(
            F.count("*").alias("sessions"),
            F.round(F.count("*") / n_sessions * 100, 2).alias("pct"),
            F.avg("n_triggers").alias("triggers_moy"),
            F.avg("n_session_products").alias("produits_vus_moy"),
            F.avg(f"n_found_{arm}").alias("triggers_trouvant_moy"),
            F.avg(F.col("any_model")).alias("pct_model_found"),
        )
        .orderBy(f"sess_{arm}")
        .show(truncate=False)
    )

print("===== croisement display x relevance, au niveau SESSION =====")
sessions_level.groupBy("sess_display").pivot("sess_relevance", ["positive", "full_negative"]).count().show()